In [1]:
import numpy as np
from core_methodology import CORE

### Пример портфеля

In [2]:
# задан портфель из двух инструментов
portfolio = [
    {
        'type': 'stock',
        'underlying': 'stock1',
        'ds': False,

        'quantity': 1,
        'daily_limit': 10_000,
        'execution_lag': 3
    },
    {
        'type': 'stock',
        'underlying': 'stock1',
        'ds': False,

        'quantity': -2,
        'daily_limit': 10_000,
        'execution_lag': 1
    }
]

#### Риск-сценарии $\mathbf{r_k}$
Они же ценовые пути, price_paths $P_i(\mathbf{r_k})$

Генерация производится непосредственно до реализации методологии CORE, в ней необходимы только ценовые пути

In [3]:
spots = [100, 100]

In [4]:
# временная функция, присобачиваем Q_{i,0} к price_path
def stack_prices(portfolio, price_path):
    def_price = np.array(spots).reshape(len(portfolio), 1)

    return np.hstack((def_price, price_path))

In [5]:
price_path1 = np.array([[130, 150, 160], # цена 1-го актива в течении времени
                         [130, 150, 160]]) # цена 2-го актива в течении времени
price_path1 = stack_prices(portfolio, price_path1)

price_path2 = np.array([[80, 70, 60], # цена 1-го актива в течении времени
                         [80, 70, 60]]) # цена 2-го актива в течении времени
price_path2 = stack_prices(portfolio, price_path2)

price_path3 = np.array([[110, 60, 80], # цена 1-го актива в течении времени
                        [110, 60, 80]]) # цена 2-го актива в течении времени
price_path3 = stack_prices(portfolio, price_path3)

In [6]:
price_paths = np.array([price_path1,
                        price_path2,
                        price_path3])

for pp in price_paths:
    print(pp, end='\n\n')

[[100 130 150 160]
 [100 130 150 160]]

[[100  80  70  60]
 [100  80  70  60]]

[[100 110  60  80]
 [100 110  60  80]]



In [7]:
core = CORE(portfolio, price_paths)
core.__dict__

{'_portfolio': [{'type': 'stock',
   'underlying': 'stock1',
   'ds': False,
   'quantity': 1,
   'daily_limit': 10000,
   'execution_lag': 3,
   'days_to_maturity': 8},
  {'type': 'stock',
   'underlying': 'stock1',
   'ds': False,
   'quantity': -2,
   'daily_limit': 10000,
   'execution_lag': 1,
   'days_to_maturity': 8}],
 '_quantities': array([[ 1],
        [-2]]),
 '_ds': array([[False],
        [False]]),
 '_abs_quantities': array([[1],
        [2]]),
 '_price_paths': array([[[100, 130, 150, 160],
         [100, 130, 150, 160]],
 
        [[100,  80,  70,  60],
         [100,  80,  70,  60]],
 
        [[100, 110,  60,  80],
         [100, 110,  60,  80]]]),
 '_psi': array([[[ 130,  150,  160],
         [-260, -300, -320]],
 
        [[  80,   70,   60],
         [-160, -140, -120]],
 
        [[ 110,   60,   80],
         [-220, -120, -160]]]),
 '_T': 3,
 '_strategy': None,
 '_pl_matrix': None,
 '_c_value': None}

In [8]:
opt_strategy = core.optimized_strategy()

Set parameter Username
Academic license - for non-commercial use only - expires 2022-08-03
Gurobi Optimizer version 9.5.1 build v9.5.1rc2 (mac64[arm])
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 19 rows, 7 columns and 77 nonzeros
Model fingerprint: 0x9f2b04f7
Variable types: 1 continuous, 6 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+00]
  RHS range        [1e+00, 1e+04]
Presolve removed 14 rows and 3 columns
Presolve time: 0.00s
Presolved: 5 rows, 4 columns, 15 nonzeros
Variable types: 0 continuous, 4 integer (0 binary)
Found heuristic solution: objective -320.0000000
Found heuristic solution: objective -300.0000000

Root relaxation: objective -2.600000e+02, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap 

In [9]:
print(opt_strategy)
print(core.get_pl_matrix())
print(core.get_c_value())

[[0 0 1]
 [1 0 1]]
[[[   0.    0.  160.]
  [-130.    0. -160.]]

 [[   0.    0.   60.]
  [ -80.    0.  -60.]]

 [[   0.    0.   80.]
  [-110.    0.  -80.]]]
130.0


In [10]:
strats_for_1 = [0, 0, 1]
strats_for_2 = [[2, 0, 0],
                [1, 1, 0],
                [1, 0, 1],
                [0, 1, 1],
                [0, 2, 0],
                [0, 0, 2]]

In [11]:
for j in strats_for_2:
    strategy = np.array([strats_for_1, j])
    core.set_strategy(new_strategy=strategy)
    print(strategy, core.get_c_value(), sep='\t')

[[0 0 1]
 [2 0 0]]	260.0
[[0 0 1]
 [1 1 0]]	280.0
[[0 0 1]
 [1 0 1]]	130.0
[[0 0 1]
 [0 1 1]]	150.0
[[0 0 1]
 [0 2 0]]	300.0
[[0 0 1]
 [0 0 2]]	160.0


In [12]:
print(core.naive_strategy())
print(core.get_pl_matrix())
print(core.get_c_value())

[[0 0 1]
 [2 0 0]]
[[[   0.    0.  160.]
  [-260.    0.    0.]]

 [[   0.    0.   60.]
  [-160.    0.    0.]]

 [[   0.    0.   80.]
  [-220.    0.    0.]]]
260.0
